[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_17/2_sql.ipynb)

# Day 17: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (32 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | The shopping funnel at a glance | medium | 5 |
| Q2 | A funnel that respects the order | medium | 8 |
| Q3 | Carts that turn into orders within a day | medium | 7 |
| Q4 | Clean the experiment before reading it (review) | medium | 6 |
| Q5 | Is an early cart an activation signal? (review) | medium | 6 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: Udacity landing-page A/B test log (educational use; has deliberate data problems).
pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv")).rename(columns={"group": "grp"}).to_sql("ab_data", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: ab_data** (Udacity landing-page experiment, January 2017; contains deliberate mistakes):

| Table | One row is | Columns |
|---|---|---|
| `ab_data` | one page visit (294,478) | `user_id`, `timestamp` (text with microseconds), `grp` ('control' / 'treatment'; renamed from `group`, a reserved word), `landing_page` ('old_page' / 'new_page'), `converted` (0/1) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. The shopping funnel at a glance

*medium, about 5 min*

Shopping app (made up). The funnel steps are, in order: 1 `view_item`, 2 `add_to_cart`, 3 `checkout`,
4 `purchase`. For each step return `step_no`, `step`, `users` (distinct users who did that event at least once, at any
time), `pct_of_first` (users as a percent of step 1) and `pct_of_prev` (percent of the previous step), 1 decimal.

Example shape: `2 | add_to_cart | 1091 | 79.1 | 79.1`.

Follow-up: this version ignores order and timing. Name two ways it can overstate conversion.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "how many users reach each step". Pattern: a small step table (to fix the order and keep empty steps), COUNT(DISTINCT user_id) per step, then FIRST_VALUE / LAG for the percentages.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE steps with `SELECT 1 AS step_no, 'view_item' AS step UNION ALL ...`.
2. LEFT JOIN app_events on event_name, `COUNT(DISTINCT e.user_id)` per step.
3. `FIRST_VALUE(users) OVER (ORDER BY step_no)` and `LAG(users) OVER (ORDER BY step_no)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH steps AS (
    SELECT 1 AS step_no, 'view_item' AS step
    UNION ALL SELECT 2, 'add_to_cart'
    UNION ALL SELECT 3, 'checkout'
    UNION ALL SELECT 4, 'purchase'
), reached AS (
    SELECT s.step_no, s.step, COUNT(DISTINCT e.user_id) AS users
    FROM steps s
    LEFT JOIN app_events e ON e.event_name = s.step
    GROUP BY s.step_no, s.step
)
SELECT step_no, step, users,
       ROUND(100.0 * users / FIRST_VALUE(users) OVER (ORDER BY step_no), 1) AS pct_of_first,
       ROUND(100.0 * users / LAG(users) OVER (ORDER BY step_no), 1)         AS pct_of_prev
FROM reached
ORDER BY step_no
```

Result:

| step_no | step | users | pct_of_first | pct_of_prev |
|---|---|---|---|---|
| 1 | view_item | 1380 | 100.0 | NULL |
| 2 | add_to_cart | 1091 | 79.1 | 79.1 |
| 3 | checkout | 873 | 63.3 | 80.0 |
| 4 | purchase | 422 | 30.6 | 48.3 |

**Why:** The step table gives the funnel a fixed order (alphabetical order would be wrong) and, with a LEFT JOIN, shows a step with 0 users instead of hiding it. The biggest drop is checkout to purchase (48.3%). Follow-up: (1) a user can add to cart without viewing first (from a wishlist), or purchase months after an unrelated view, and still counts; (2) there is no time window, so old users have had more time to convert than new ones. Q2 and Q3 fix both.

**Complexity:** One pass over app_events with a group by on 4 steps.

**Common mistakes:** Counting events instead of distinct users (a user with 5 carts counts 5 times). Ordering steps alphabetically. Integer division.

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
### Q2. A funnel that respects the order

*medium, about 8 min*

Shopping app. Now a user only counts for a step if they did it **after** the previous step:
- viewed: has a `view_item` (take their first view time);
- carted: has an `add_to_cart` **after** that first view (take the first such cart time);
- checked_out: a `checkout` after that cart time; purchased: a `purchase` after that checkout time.

Per `platform` (all users, also those who never viewed) return `viewed`, `carted`, `checked_out`, `purchased` and
`view_to_purchase_pct` (purchased / viewed, 1 decimal).

Example: a user with add_to_cart at 09:00, view_item at 09:05 and nothing else counts as viewed but not carted.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "step B must happen after step A". Pattern: a chain of CTEs, each taking the first time of a step that is later than the previous step's time (join + MIN), then LEFT JOIN all to users.

</details>

<details><summary><b>Hint 2</b></summary>

1. v: MIN(event_time) of view_item per user.
2. c: join app_events to v on user, `event_name = 'add_to_cart' AND e.event_time > v.t`, MIN per user.
3. k from c, p from k the same way.
4. app_users LEFT JOIN v, c, k, p; COUNT(x.user_id) per platform.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH v AS (
    SELECT user_id, MIN(event_time) AS t
    FROM app_events WHERE event_name = 'view_item'
    GROUP BY user_id
), c AS (
    SELECT e.user_id, MIN(e.event_time) AS t
    FROM app_events e JOIN v ON v.user_id = e.user_id
    WHERE e.event_name = 'add_to_cart' AND e.event_time > v.t
    GROUP BY e.user_id
), k AS (
    SELECT e.user_id, MIN(e.event_time) AS t
    FROM app_events e JOIN c ON c.user_id = e.user_id
    WHERE e.event_name = 'checkout' AND e.event_time > c.t
    GROUP BY e.user_id
), p AS (
    SELECT e.user_id, MIN(e.event_time) AS t
    FROM app_events e JOIN k ON k.user_id = e.user_id
    WHERE e.event_name = 'purchase' AND e.event_time > k.t
    GROUP BY e.user_id
)
SELECT u.platform,
       COUNT(v.user_id) AS viewed,
       COUNT(c.user_id) AS carted,
       COUNT(k.user_id) AS checked_out,
       COUNT(p.user_id) AS purchased,
       ROUND(100.0 * COUNT(p.user_id) / COUNT(v.user_id), 1) AS view_to_purchase_pct
FROM app_users u
LEFT JOIN v ON v.user_id = u.user_id
LEFT JOIN c ON c.user_id = u.user_id
LEFT JOIN k ON k.user_id = u.user_id
LEFT JOIN p ON p.user_id = u.user_id
GROUP BY u.platform
ORDER BY u.platform
```

Result:

| platform | viewed | carted | checked_out | purchased | view_to_purchase_pct |
|---|---|---|---|---|---|
| android | 642 | 519 | 421 | 205 | 31.9 |
| ios | 545 | 431 | 348 | 176 | 32.3 |
| web | 193 | 133 | 104 | 41 | 21.2 |

**Why:** Each CTE has at most one row per user, so the final LEFT JOINs cannot multiply rows and COUNT(x.user_id) counts users who reached step x in order. Taking the first time at each step is the standard "greedy" choice: it gives every user the most room to complete the next step. Web converts clearly worse (21.2% versus about 32%), so look at the web checkout first. Timestamps in 'YYYY-MM-DD HH:MM:SS' compare correctly as text in SQLite; in PostgreSQL they are real timestamps.

**Complexity:** One group by per step; each join is on user_id.

**Common mistakes:** Joining all events to all events (one row per combination: huge and double counted). Comparing with the previous step's LAST time instead of the first. Using `>=` when two steps can share a timestamp only by logging error (decide and say it).

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
### Q3. Carts that turn into orders within a day

*medium, about 7 min*

Shopping app. For every `add_to_cart` event, check whether the **same user** has a `purchase` event
**after** it and **at most 24 hours** later. Per `platform` return `carts` (cart events), `converted_24h` and
`abandonment_pct` (carts without a purchase in 24 hours, 1 decimal).

Example: cart at 2024-01-05 21:10:00 and purchase at 2024-01-06 20:00:00 converts; purchase at 2024-01-06 21:30:00
does not.

Follow-up: two carts at 10:00 and 10:20 followed by one purchase at 10:30 both count as converted. Is that what
the PM wants?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "followed by X within T". Pattern: a correlated EXISTS (or LEFT JOIN with a time range) per cart event.

</details>

<details><summary><b>Hint 2</b></summary>

1. For each cart e: `EXISTS (SELECT 1 FROM app_events p WHERE p.user_id = e.user_id AND p.event_name = 'purchase' AND p.event_time > e.event_time AND p.event_time <= datetime(e.event_time, '+24 hours'))`.
2. Group by platform: COUNT(*) and SUM of the flag.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH carts AS (
    SELECT e.event_id, u.platform,
           EXISTS (
               SELECT 1
               FROM app_events p
               WHERE p.user_id = e.user_id
                 AND p.event_name = 'purchase'
                 AND p.event_time > e.event_time
                 AND p.event_time <= datetime(e.event_time, '+24 hours')
           ) AS bought_24h
    FROM app_events e
    JOIN app_users u ON u.user_id = e.user_id
    WHERE e.event_name = 'add_to_cart'
)
SELECT platform,
       COUNT(*)        AS carts,
       SUM(bought_24h) AS converted_24h,
       ROUND(100.0 * (COUNT(*) - SUM(bought_24h)) / COUNT(*), 1) AS abandonment_pct
FROM carts
GROUP BY platform
ORDER BY platform
```

Result:

| platform | carts | converted_24h | abandonment_pct |
|---|---|---|---|
| android | 2096 | 490 | 76.6 |
| ios | 2030 | 472 | 76.7 |
| web | 512 | 115 | 77.5 |

**Why:** EXISTS answers yes or no per cart without multiplying rows, even if several purchases follow. In SQLite EXISTS returns 1 or 0, so SUM counts conversions; in PostgreSQL it returns a boolean, use `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` or `COUNT(*) FILTER (WHERE bought_24h)`. Abandonment is about 77% on every platform, so the web gap from Q2 is not at the cart step. Follow-up: counting at cart level credits one purchase to two carts. A PM usually wants user-day or session level ("did this session end in a purchase?"), or to attribute each purchase to the latest cart before it (LEAD / MAX).

**Complexity:** One correlated lookup per cart; with an index on (user_id, event_name, event_time) it is fast.

**Common mistakes:** A LEFT JOIN to purchases with the time range, then COUNT(*) (a cart followed by two purchases counts twice). Forgetting `p.event_time > e.event_time` (purchases before the cart count). Using date() and losing the hours.

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-subqueries](https://mahsa-agz.github.io/ds-handbook/#sql-subqueries)

</details>

---
### Q4. Clean the experiment before reading it (review)

*medium, about 6 min*

`ab_data` (real Udacity experiment): control users should see the `old_page`, treatment users the
`new_page`. Some rows break this rule. Per `grp` return `n_rows`, `mismatched` (rows where group and page disagree),
`clean_users` (distinct users among rows that agree) and `conv_pct_clean` (conversion rate on the rows that agree,
2 decimals).

Example: a row `treatment | old_page` is mismatched and must not enter the conversion rate.

Then say whether the new page wins.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: CASE WHEN inside aggregates (conditional counts and a conditional average). A neat trick: `(grp = 'treatment') = (landing_page = 'new_page')` is true exactly when the row is consistent.

</details>

<details><summary><b>Hint 2</b></summary>

1. `SUM(CASE WHEN <mismatch> THEN 1 ELSE 0 END)`.
2. `COUNT(DISTINCT CASE WHEN <consistent> THEN user_id END)`.
3. `AVG(CASE WHEN <consistent> THEN converted END)`: no ELSE, so mismatched rows become NULL and AVG skips them.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT grp,
       COUNT(*) AS n_rows,
       SUM(CASE WHEN (grp = 'treatment') <> (landing_page = 'new_page') THEN 1 ELSE 0 END) AS mismatched,
       COUNT(DISTINCT CASE WHEN (grp = 'treatment') = (landing_page = 'new_page') THEN user_id END) AS clean_users,
       ROUND(100.0 * AVG(CASE WHEN (grp = 'treatment') = (landing_page = 'new_page') THEN converted END), 2)
           AS conv_pct_clean
FROM ab_data
GROUP BY grp
```

Result:

| grp | n_rows | mismatched | clean_users | conv_pct_clean |
|---|---|---|---|---|
| control | 147202 | 1928 | 145274 | 12.04 |
| treatment | 147276 | 1965 | 145310 | 11.88 |

**Why:** Comparing two booleans is a compact XOR test. Leaving out ELSE in the AVG is the key trick: NULLs are ignored, so the average is over clean rows only (ELSE 0 would wrongly count mismatches as non-converted). About 1.3% of rows are mismatched in each group. Treatment has one more clean row than clean users (a duplicated user): deduplicate before testing. Clean conversion is 12.04% for control and 11.88% for treatment: the new page does not win (it is slightly lower; a z-test is the stats notebook's job).

**Complexity:** One pass over 294,478 rows.

**Common mistakes:** `ELSE 0` in the conversion average. Counting rows as users. Analysing without checking assignment consistency first. `group` is a reserved word in SQL: quote it or rename it (done here: `grp`).

**Learn more:** [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis)

</details>

---
### Q5. Is an early cart an activation signal? (review)

*medium, about 6 min*

Shopping app. Product believes users who **add to cart on their signup day** stick around. Split users
into 'carted on day 0' and 'did not cart on day 0' and return `segment`, `users` and `d7_pct` (percent active exactly
7 days after signup, 1 decimal).

What do you tell Product?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: per user, two EXISTS flags (did the action on day 0; active on day 7), then a group by on the first flag with AVG of the second.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE over app_users with `EXISTS (... add_to_cart AND substr(event_time, 1, 10) = signup_date)` and `EXISTS (... activity_date = date(signup_date, '+7 day'))`.
2. CASE to label, group by it.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH day0 AS (
    SELECT u.user_id,
           EXISTS (SELECT 1 FROM app_events e
                   WHERE e.user_id = u.user_id AND e.event_name = 'add_to_cart'
                     AND substr(e.event_time, 1, 10) = u.signup_date) AS carted_day0,
           EXISTS (SELECT 1 FROM daily_activity d
                   WHERE d.user_id = u.user_id AND d.activity_date = date(u.signup_date, '+7 day')) AS d7
    FROM app_users u
)
SELECT CASE WHEN carted_day0 = 1 THEN 'carted on day 0' ELSE 'did not cart on day 0' END AS segment,
       COUNT(*) AS users,
       ROUND(100.0 * AVG(d7), 1) AS d7_pct
FROM day0
GROUP BY CASE WHEN carted_day0 = 1 THEN 'carted on day 0' ELSE 'did not cart on day 0' END
```

Result:

| segment | users | d7_pct |
|---|---|---|
| carted on day 0 | 421 | 28.0 |
| did not cart on day 0 | 1079 | 30.3 |

**Why:** EXISTS flags keep one row per user, so AVG(d7) is a clean retention rate per segment. Here the early cart users retain slightly worse (28.0% versus 30.3%), so in this data a day-0 cart is not an activation signal. Tell Product: test other first-day actions (number of sessions, minutes), and remember that even a strong correlation would not prove that pushing users to cart causes retention; that needs an experiment.

**Complexity:** Two correlated lookups per user.

**Common mistakes:** Joining events directly (users with several carts get several rows and dominate the average). Comparing a timestamp to a date without truncating. Claiming causation.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [stats-product-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-product-metrics)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_17/3_stats.ipynb)